In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
root = os.getcwd()
root

'/home/peiretti/fair-clustering'

In [4]:
plot_path = root + "/plots"
plot_path

'/home/peiretti/fair-clustering/plots'

In [5]:
def create_path(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"Folder '{path}' created successfully.")
    else:
        print(f"Folder '{path}' already exists.")

In [6]:
clusters = 3
groups = 2

In [9]:
taucc_path = root + f"/results/synthetic/clus{clusters}_rc"
filter_sensitive_degree = "sensitive_px==0.8 and sensitive_py==0.8"
taucc_fair_max = pd.read_csv(taucc_path + f"/taucc_fair_max/topsis_best_param_groups{groups}_rc.csv").query(filter_sensitive_degree)
taucc_fair = pd.read_csv(taucc_path + f"/taucc_fair/topsis_best_param_groups{groups}_rc.csv").query(filter_sensitive_degree)
taucc_vanilla = pd.read_csv(taucc_path + f"/taucc_vanilla/aggregated_groups{groups}.csv").query(filter_sensitive_degree)

In [10]:
taucc_fair

,sensitive_px,sensitive_py,row_fair_majority,row_fair_minority,col_fair_majority,col_fair_minority,tau_x_mean,tau_x_std,tau_x_var,tau_y_mean,...,AMI_van_cols_var,ARI_van_cols_mean,ARI_van_cols_std,ARI_van_cols_var,time_mean,time_std,time_var,balance_bera_mean,balance_bera_std,balance_bera_var
0,0.8,0.8,0.25,0.75,0.5,0.0,0.39502,0.0,0.0,0.39502,...,0.0,1.0,0.0,0.0,2.441322,0.814376,0.663208,0.514226,0.0,0.0


In [11]:
cols_to_check = ["tau_x_mean", "tau_x_std", "tau_y_mean", "tau_y_std", "ARI_rows_mean", "ARI_rows_std", "ARI_cols_mean", "ARI_cols_std",  "ARI_van_rows_mean", "ARI_van_rows_std", "ARI_van_cols_mean", "ARI_van_cols_std", "balance_bera_mean", "balance_bera_std", "time_mean", "time_std"]

dfs = {
    'taucc_vanilla': taucc_vanilla, 
    'taucc_fair': taucc_fair, 
    'taucc_fair_max': taucc_fair_max
}

summary = pd.DataFrame(
    {name: [col in df.columns for col in cols_to_check] for name, df in dfs.items()},
    index=cols_to_check
)
summary



,taucc_vanilla,taucc_fair,taucc_fair_max
tau_x_mean,True,True,True
tau_x_std,True,True,True
tau_y_mean,True,True,True
tau_y_std,True,True,True
ARI_rows_mean,True,True,True
ARI_rows_std,True,True,True
ARI_cols_mean,True,True,True
ARI_cols_std,True,True,True
ARI_van_rows_mean,False,True,True
ARI_van_rows_std,False,True,True


In [12]:
rows = []
for algo, df in dfs.items():
    row = {"algorithm": algo}
    cols = ["tau_x", "tau_y", "ARI_rows", "ARI_cols", "ARI_van_rows", "ARI_van_cols", "balance_bera", "time"]
    for col in cols:
        if f"{col}_mean" in df.columns:
            row[f"{col}_mean"] = df[f"{col}_mean"].values[0]
            row[f"{col}_std"] = df[f"{col}_std"].values[0]
        else:
            row[f"{col}_mean"] = None
            row[f"{col}_std"] = None
    rows.append(row)

result = pd.DataFrame(rows, columns=[
    "algorithm", "tau_x_mean", "tau_x_std", "tau_y_mean", "tau_y_std",
    "ARI_rows_mean", "ARI_rows_std","ARI_cols_mean", "ARI_cols_std", 
    "ARI_van_rows_mean", "ARI_van_rows_std","ARI_van_cols_mean", "ARI_van_cols_std", 
    "balance_bera_mean", "balance_bera_std",
    "time_mean", "time_std"
])

In [13]:
result

,algorithm,tau_x_mean,tau_x_std,tau_y_mean,tau_y_std,ARI_rows_mean,ARI_rows_std,ARI_cols_mean,ARI_cols_std,ARI_van_rows_mean,ARI_van_rows_std,ARI_van_cols_mean,ARI_van_cols_std,balance_bera_mean,balance_bera_std,time_mean,time_std
0,taucc_vanilla,0.395020,0.000000,0.395020,0.000000,0.528719,0.0,0.564064,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,0.611734,0.193277
1,taucc_fair,0.395020,0.000000,0.395020,0.000000,0.528719,0.0,0.564064,0.000000,1.000000,0.0,1.000000,0.0,0.514226,0.0,2.441322,0.814376
2,taucc_fair_max,0.325283,0.025283,0.325283,0.025283,0.420829,0.0,0.457941,0.017346,0.602604,0.0,0.664514,0.0,0.894390,0.0,1.474832,0.729267


In [20]:
print(taucc_fair["balance_bera_rows_std"])
print(taucc_fair["balance_bera_cols_std"])

KeyError: 'balance_bera_rows_std'

In [11]:
result.to_csv(root + f"/results/clus{clusters}_groups{groups}.csv", index=False)

# Plot - Fair TauCC vs competitors

In [ ]:
# Fairness
algo_names = list(dfs.keys())[::-1]
means = [np.round(df['balance_bera_mean'].values[0], 3) for df in dfs.values()][::-1]
stds  = [np.round(df['balance_bera_std'].values[0],  3) for df in dfs.values()][::-1]

x = np.arange(len(algo_names))
width = 0.5

fig, ax = plt.subplots(figsize=(7, 5), layout='constrained')

bars = ax.barh(x, means, width, xerr=stds, capsize=5, color='#EF9A9A')
ax.bar_label(bars, labels=[f'{m}' for m in means], padding=5, fontsize=10)

ax.set_title('Fairness evaluation')
ax.set_xlabel('Balance')
ax.set_yticks(x)
ax.set_yticklabels(algo_names)
ax.set_xlim(0, 1.10)
plt.savefig(plot_path + f"/fairness.png",
            dpi=300,
            transparent=True,
            bbox_inches='tight')
#plt.show()

In [ ]:
# Time
algo_names = list(dfs.keys())[::-1]
time_means = [np.round(df['time_mean'].values[0], 2) for df in dfs.values()][::-1]
time_stds  = [np.round(df['time_std'].values[0], 2) for df in dfs.values()][::-1]

x = np.arange(len(algo_names))
width = 0.5

fig, ax = plt.subplots(figsize=(7, 5), layout='constrained')

bars = ax.barh(x, time_means, width, xerr=time_stds, capsize=5, color='#00838F')
ax.bar_label(bars, labels=[f'{m}' for m in time_means], padding=5, fontsize=10)

ax.set_title('Execution time (in seconds)')
ax.set_xlabel('Time (sec)')
ax.set_yticks(x)
ax.set_yticklabels(algo_names)
max_val = max(m + s for m, s in zip(time_means, time_stds))
ax.set_xlim(0, max_val * 1.2)
plt.savefig(plot_path + f"/time.png",
            dpi=300,
            transparent=True,
            bbox_inches='tight')
#plt.show()

In [ ]:
algo_names = list(dfs.keys())[::-1]

def plot_metrics(metrics, colors, title, filename, name_xlabel, legend_loc='best'):
    n_algos   = len(algo_names)
    n_metrics = len(metrics)
    width     = 0.35
    x         = np.arange(n_algos)

    fig, ax = plt.subplots(figsize=(8, 6), layout='constrained')

    # Tieni solo gli algoritmi che hanno almeno una delle metriche
    filtered = [(name, dfs[name]) for name in algo_names
                if any(f'{m}_mean' in dfs[name].columns for m in metrics)]
    algo_names_filtered = [name for name, _ in filtered]
    x = np.arange(len(algo_names_filtered))

    for i, (metric, color) in enumerate(zip(metrics, colors)):
        means, stds = [], []
        for name, df in filtered:
            if f'{metric}_mean' in df.columns:
                means.append(np.round(df[f'{metric}_mean'].values[0], 3))
                stds.append(np.round(df[f'{metric}_std'].values[0],  3))
            else:
                means.append(np.nan)
                stds.append(np.nan)

        valid_means = [m if not np.isnan(m) else 0 for m in means]
        valid_stds  = [s if not np.isnan(s) else 0 for s in stds]

        #offset = (i - n_metrics / 2 + 0.5) * width
        offset = (n_metrics / 2 - i - 0.5) * width
        bars = ax.barh(x + offset, valid_means, width, xerr=valid_stds,
                       capsize=4, color=color, label=metric)
        ax.bar_label(bars, labels=[f'{m:.3f}' if not np.isnan(m) else '' for m in means],
                     padding=5, fontsize=10)

    ax.set_title(title)
    ax.set_xlabel(name_xlabel)
    ax.set_yticks(x)
    ax.set_yticklabels(algo_names_filtered)
    ax.legend(loc=legend_loc)
    
    max_val = max((m + s for m, s in zip(valid_means, valid_stds) if m > 0), default=1)
    ax.set_xlim(0, max_val * 1.10)

    plt.savefig(plot_path + f"/{filename}.png",
                dpi=300,
                transparent=True,
                bbox_inches='tight')
    #plt.show()


In [ ]:
# tau_x e tau_y
plot_metrics(
    metrics=['tau_x', 'tau_y'],
    colors=['#2196F3', '#90CAF9'],
    title='Clustering Quality (tau x, tau y)',
    filename='taus',
    name_xlabel='taus',
    legend_loc='lower right'
)
    
# ARI_rows e ARI_cols
plot_metrics(
    metrics=['ARI_rows', 'ARI_cols'],
    colors=['#FF6F00', '#FFCC80'],
    title='Clustering Quality (ARI w.r.t. vanilla)',
    filename='ari_rows_cols',
    name_xlabel='ARI'
)

# ARI_true_labels
plot_metrics(
    metrics=['ARI_true_labels'],
    colors=['#43A047'],
    title='Clustering Quality (ARI w.r.t. true labels)',
    filename='ari_true_labels',
    name_xlabel='ARI'
)

"""
# NMI_rows e NMI_cols
plot_metrics(
    metrics=['NMI_rows', 'NMI_cols'],
    colors=['tomato', 'seagreen'],
    title='Clustering Quality (NMI)',
    filename='nmi_rows_cols',
    name_xlabel='NMI'
)

# NMI_true_labels
plot_metrics(
    metrics=['NMI_true_labels'],
    colors=['steelblue'],
    title='Clustering Quality (NMI true labels)',
    filename='nmi_true_labels',
    name_xlabel='NMI'
)
"""


# Old code

In [ ]:
taucc_path = root + f"/results/{dataset}/{sensitive}/taucc_{version}_old/init_random"
frisch_path = root + f"/algorithms/C-Fairness-RecSys/reproducibility_study/Frisch_et_al/results/{dataset}/{sensitive}/lbm_{version}"

In [ ]:
taucc = pd.read_csv(taucc_path + "/aggregated.csv")
frisch = pd.read_csv(frisch_path + "/aggregated.csv")

In [ ]:
taucc_vanilla_path = root + f"/results/{dataset}/{sensitive}/taucc_vanilla/init_random"
taucc_vanilla = pd.read_csv(taucc_vanilla_path + "/results_aggregated_10.csv")

frisch_vanilla_path = root + f"/algorithms/C-Fairness-RecSys/reproducibility_study/Frisch_et_al/results/{dataset}/{sensitive}/lbm_baseline"
frisch_vanilla = pd.read_csv(frisch_vanilla_path + "/aggregated.csv")

In [ ]:
taucc

In [ ]:
fair_taucc = taucc[(taucc["fair_major"]==1.0) & (taucc["fair_minor"]==1.0)]
#fair_taucc = taucc[(taucc["fair_major"]==1.0) & (taucc["fair_minor1"]==1.0) & (taucc["fair_minor2"]==1.0)]
fair_taucc

In [ ]:
taucc_relax = taucc[(taucc["fair_major"]==0.9) & (taucc["fair_minor"]==1.0)]
#taucc_relax = taucc[(taucc["fair_major"]==0.9) & (taucc["fair_minor1"]==0.9) & (taucc["fair_minor2"]==1.0)]
taucc_relax

In [ ]:
"""
mask_col1 = taucc['fair_major'].isin([0.9, 1.0])
mask_col2 = taucc['fair_minor'].isin([0.9, 1.0])
mask_different = taucc["fair_major"] != taucc["fair_minor"]
taucc_relax = taucc[mask_col1 & mask_col2 & mask_different]
taucc_relax
"""


"""
mask_col1 = taucc['fair_major'].isin([0.9, 1.0])
mask_col2 = taucc['fair_minor1'].isin([0.9, 1.0])
mask_col3 = taucc['fair_minor2'].isin([0.9, 1.0])
taucc_relax = taucc[mask_col1 & mask_col2 & mask_col3]
taucc_relax
"""

In [ ]:
"""
index_max = taucc_relax['tau_x_mean'].idxmax()
row_max = taucc_relax.loc[index_max]
taucc_relax = pd.DataFrame([row_max])
taucc_relax
"""

In [ ]:
taucc_vanilla = taucc_vanilla.rename(columns={
    'ARI_mean':'ARI_true_labels_mean', 
    'ARI_std': 'ARI_true_labels_std',
    'ARI_var': 'ARI_true_labels_var',
    'AMI_mean':'AMI_true_labels_mean', 
    'AMI_std': 'AMI_true_labels_std',
    'AMI_var': 'AMI_true_labels_var',
    'NMI_mean':'NMI_true_labels_mean', 
    'NMI_std': 'NMI_true_labels_std',
    'NMI_var': 'NMI_true_labels_var'
})


In [ ]:
df = pd.concat([taucc_vanilla, fair_taucc, taucc_relax, frisch_vanilla, frisch], ignore_index=True)
df

In [ ]:
df.insert(0, "algorithm", ["TauCC", "Fair TauCC", "Fair TauCC weak", "LBM", "Parity LBM"])
df

In [ ]:
#res_all = df[["algorithm", "tau_x_mean", "tau_x_std", "tau_y_mean", "tau_y_std", "ARI_true_labels_mean", "ARI_true_labels_std", "ARI_rows_mean", "ARI_rows_std", "ARI_cols_mean", "ARI_cols_std", "balance_bera_mean", "balance_bera_std", "KL_fairness_error_mean", "KL_fairness_error_std"]]
res_all = df[["algorithm", "time_mean", "time_std"]]

In [ ]:
res_all

In [ ]:
res_table = root + f"/results/{dataset}_{sensitive}.csv"

In [ ]:
res_all.to_csv(res_table, sep=";", index=False)

## Fairness

In [ ]:
algo = df["algorithm"]
metrics_labels = ["balance_bera", "balance_chierichetti", "KL_fairness_error"]

metrics = {
    "Fair TauCC": [],
    "Parity LBM": []
}

metrics_std = {
    "Fair TauCC": [],
    "Parity LBM": []
}

for label in metrics_labels:
    
    metrics["Fair TauCC"].append(
        np.round(taucc[label+"_mean"].values[0], 2)
    )
    metrics_std["Fair TauCC"].append(
        np.round(taucc[label+"_std"].values[0], 2)
    )
    
    metrics["Parity LBM"].append(
        np.round(frisch[label+"_mean"].values[0], 2)
    )
    metrics_std["Parity LBM"].append(
        np.round(frisch[label+"_std"].values[0], 2)
    )
    
print(metrics)

x = np.arange(len(metrics_labels))  # the label locations
width = 0.20  # the width of the bars
multiplier = -0.5

fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in metrics.items():
    offset = width * multiplier
    errors = metrics_std[attribute]
    #rects = ax.bar(x + offset, measurement, width, label=attribute, yerr=errors, capsize=5)
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title('Results on Fairness')
ax.set_xticks(x + width, metrics_labels)
ax.legend(loc='upper right')

#plt.show()
plt.savefig(plot_path + f"/{dataset}_{sensitive}_fairness.png")

## NMI, AMI, ARI w.r.t. true labels

In [ ]:
algo = df["algorithm"]
metrics_labels = ["NMI_true_labels", "ARI_true_labels", "AMI_true_labels"]
    

metrics = {
    "Fair TauCC": [],
    "Parity LBM": [],
}

metrics_std = {
    "Fair TauCC": [],
    "Parity LBM": [],
}

for label in metrics_labels:
    
    metrics["Fair TauCC"].append(
        np.round(taucc[label+"_mean"].values[0], 2)
    )
    metrics_std["Fair TauCC"].append(
        np.round(taucc[label+"_std"].values[0], 2)
    )
    
    metrics["Parity LBM"].append(
        np.round(frisch[label+"_mean"].values[0], 2)
    )
    metrics_std["Parity LBM"].append(
        np.round(frisch[label+"_std"].values[0], 2)
    )

print(metrics)

x = np.arange(len(metrics_labels))  # the label locations

if dataset == "lfw":
    width = 0.20  # the width of the bars
    multiplier = -0.5
else:
    width = 0.25  # the width of the bars
    multiplier = 0.5

fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in metrics.items():
    offset = width * multiplier
    errors = metrics_std[attribute]
    #rects = ax.bar(x + offset, measurement, width, label=attribute, yerr=errors, capsize=5)
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title('Results on clustering quality (fair vs true labels)')
ax.set_xticks(x + width, metrics_labels)
ax.legend(loc='upper center')

#plt.show()
plt.savefig(plot_path + f"/{dataset}_{sensitive}_true_labels.png")

## NMI, ARI, AMI w.r.t. vanilla rows

In [ ]:
algo = df["algorithm"]
metrics_labels = ["NMI_rows", "ARI_rows", "AMI_rows"]

metrics = {
    "Fair TauCC": [],
    "Parity LBM": []
}

metrics_std = {
    "Fair TauCC": [],
    "Parity LBM": []
}

for label in metrics_labels:
    
    metrics["Fair TauCC"].append(
        np.round(taucc[label+"_mean"].values[0], 2)
    )
    metrics_std["Fair TauCC"].append(
        np.round(taucc[label+"_std"].values[0], 2)
    )
    
    metrics["Parity LBM"].append(
        np.round(frisch[label+"_mean"].values[0], 2)
    )
    metrics_std["Parity LBM"].append(
        np.round(frisch[label+"_std"].values[0], 2)
    )
    

print(metrics)

x = np.arange(len(metrics_labels))  # the label locations
width = 0.20  # the width of the bars
multiplier = -0.5

fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in metrics.items():
    offset = width * multiplier
    errors = metrics_std[attribute]
    #rects = ax.bar(x + offset, measurement, width, label=attribute, yerr=errors, capsize=5)
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title('Results on Co-clustering quality (fair vs vanilla on rows)')
ax.set_xticks(x + width, metrics_labels)
ax.legend(loc='upper center')

#plt.show()
plt.savefig(plot_path + f"/{dataset}_{sensitive}_rows.png")

## NMI, ARI, AMI w.r.t. vanilla cols

In [ ]:
# Dati esemplificativi
algo = df["algorithm"]
metrics_labels = ["NMI_cols", "ARI_cols", "AMI_cols"]

metrics = {
    "Fair TauCC": [],
    "Parity LBM": []
}

metrics_std = {
    "Fair TauCC": [],
    "Parity LBM": []
}

for label in metrics_labels:
    
    metrics["Fair TauCC"].append(
        np.round(taucc[label+"_mean"].values[0], 2)
    )
    metrics_std["Fair TauCC"].append(
        np.round(taucc[label+"_std"].values[0], 2)
    )
    
    metrics["Parity LBM"].append(
        np.round(frisch[label+"_mean"].values[0], 2)
    )
    metrics_std["Parity LBM"].append(
        np.round(frisch[label+"_std"].values[0], 2)
    )
    

print(metrics)

x = np.arange(len(metrics_labels))  # the label locations
width = 0.25  # the width of the bars
multiplier = 0.5

fig, ax = plt.subplots(layout='constrained')

for attribute, measurement in metrics.items():
    offset = width * multiplier
    errors = metrics_std[attribute]
    #rects = ax.bar(x + offset, measurement, width, label=attribute, yerr=errors, capsize=5)
    rects = ax.bar(x + offset, measurement, width, label=attribute)
    ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_title('Results on clustering quality (fair vs vanilla on columns)')
ax.set_xticks(x + width, metrics_labels)
ax.legend(loc='upper right')

#plt.show()
plt.savefig(plot_path + f"/{dataset}_{sensitive}_cols.png")